In [39]:
# WRAP CODE CSS

# Source - https://stackoverflow.com/a
# Posted by Bon Ryu, modified by community. See post 'Timeline' for change history
# Retrieved 2025-12-29, License - CC BY-SA 4.0

# from IPython.display import HTML, display

# def set_css():
#   display(HTML('''
#   <style>
#     pre {
#         white-space: pre-wrap;
#     }
#   </style>
#   '''))
# get_ipython().events.register('pre_run_cell', set_css)

In [40]:
# READ EVENTS CSV

from datetime import datetime
import csv

def read_events_csv(path):
    events = {}

    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            events[row["Event_ID"]] = {
                "name": row["Event_name"],
                "type": row["Type"],
                "start": row["Start_date"],
                "end": row["End_date"]
            }

    return events


events = read_events_csv("/content/drive/MyDrive/semantic_shift/events.csv")
# events["covid-19"]["start"]

In [41]:
#READ TARGET WORDS CSV

def read_target_words_csv(path):
    pairs = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pairs.append((row["Word"], row["Event_ID"]))
    return pairs

pairs = read_target_words_csv("/content/drive/MyDrive/semantic_shift/target_words.csv")
#find event directly from word
word_to_event = dict(pairs)

In [42]:
import csv

events = read_events_csv("/content/drive/MyDrive/semantic_shift/events.csv")
target_pairs = read_target_words_csv(
    "/content/drive/MyDrive/semantic_shift/target_words.csv"
)

In [43]:
!pip install praw
import praw
import csv
from datetime import datetime
from google.colab import drive
import json

In [44]:
#secret key

# Source - https://stackoverflow.com/a
# Posted by Bob Smith
# Retrieved 2025-12-29, License - CC BY-SA 4.0

from getpass import getpass
reddit_secret_key = getpass('Enter the secret reddit key value: ')
reddit_client_id = getpass('Enter the secret reddit client id: ')

Enter the secret reddit key value: ··········
Enter the secret reddit client id: ··········


In [45]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
!pip install langdetect
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

In [47]:
#CONVERT JSON TO CSV

def json_to_csv(json_path, csv_path, word, event_id, events_dict):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    headers = ["word", "event", "register", "period", "sentence", "timestamp"]

    event_info = events_dict.get(event_id)

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(headers)

        for item in data:
            sentence = item.get("selftext", "")
            if not sentence or sentence == "[removed]" or sentence == "[deleted]":
                continue

            ts = item["created_utc"]

            timestamp = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

            period = assign_period(ts, event_info)

            writer.writerow([
                word,
                event_id,
                "informal",
                period,
                sentence.replace("\n", " "),
                timestamp
            ])


In [48]:
#REDDIT DATASET CONVERT UTC TO DESIRED FORMAT

import pandas as pd
from datetime import datetime, timezone

def extract_year(date_str):
    if not date_str or date_str == "no-change":
        return None
    try:
        parts = str(date_str).split(".")
        return int(parts[-1])
    except (ValueError, IndexError):
        return None

def assign_period(timestamp_utc, event_info):
    dt = datetime.fromtimestamp(timestamp_utc, tz=timezone.utc)
    year = dt.year

    if not event_info or event_info.get("start") == "no-change":
        return "before" if year < 2020 else "after"

    start_year = extract_year(event_info.get("start"))
    end_year = extract_year(event_info.get("end"))

    if start_year and year < start_year:
        return "before"
    if end_year and year > end_year:
        return "after"

    return "during"

In [49]:
import os, pandas as pd, praw, json
from langdetect import detect

reddit = praw.Reddit(
      client_id=reddit_client_id,
      client_secret=reddit_secret_key,
      user_agent="keyword_searcher by u/DuckApprehensive203",
  )


def get_search(query_name, event_id, max_limit, drive_path="/content/drive/MyDrive/semantic_shift/colab_outputs"):
    output_path = os.path.join(drive_path, f"reddit_{query_name}.json")
    posts = []

    print(f"Downloading: {query_name}")


    for submission in reddit.subreddit("all").search(query_name, sort="new", limit=max_limit):
      text_to_check = (submission.title or "") + " " + (submission.selftext or "")
      try:
          lang = detect(text_to_check)
      except:
          lang = "unknown"

      if lang == "en":
          post_data = {
              "id": submission.id,
              "title": submission.title,
              "subreddit": submission.subreddit.display_name,
              "score": submission.score,
              "url": submission.url,
              "created_utc": submission.created_utc,
              "author": str(submission.author),
              "num_comments": submission.num_comments,
              "selftext": submission.selftext,
          }
          posts.append(post_data)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(posts, f, indent=2, ensure_ascii=False)

    csv_output_path = output_path.replace(".json", ".csv")
    json_to_csv(output_path, csv_output_path, query_name, event_id, word_to_event)

for word, event_id in pairs:
  if event_id == 'no-change':
      event_name = "no-change"
  elif not event_id or pd.isna(event_id):
      event_name = "natural-drift"
  else:
      event_name = event_id
  get_search(word, event_name, 10000)

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Downloading: sus


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Downloading: brat


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



...

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Downloading: population


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Downloading: popular


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



In [50]:
#create zip of json and csv

import shutil

source_dir = "/content/drive/MyDrive/semantic_shift/colab_outputs"
zip_path = "/content/drive/MyDrive/semantic_shift/reddit_data.zip"

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir=source_dir
)

'/content/drive/MyDrive/semantic_shift/reddit_data.zip'